In [2]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
df_invent = spark.read.parquet("abfss://Retail_Project26@onelake.dfs.fabric.microsoft.com/Retail_LH.Lakehouse/Files/Bronze/inventory_data.parquet")

StatementMeta(, 1dc837c0-cb6e-40a0-b681-c50caec892b7, 19, Finished, Available, Finished, False)

In [3]:
display(df_invent)

StatementMeta(, 1dc837c0-cb6e-40a0-b681-c50caec892b7, 20, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 3e8fed7e-9d58-49b1-a2dc-0a5b18bba5d5)

In [4]:
df_orders = spark.read.parquet("abfss://Retail_Project26@onelake.dfs.fabric.microsoft.com/Retail_LH.Lakehouse/Files/Bronze/orders_data.parquet")
display(df_orders)

StatementMeta(, 1dc837c0-cb6e-40a0-b681-c50caec892b7, 21, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, ad3d9916-b21e-4fe4-914a-bf43a57083cb)

In [5]:
df_returns = spark.read.parquet("abfss://Retail_Project26@onelake.dfs.fabric.microsoft.com/Retail_LH.Lakehouse/Files/Bronze/returns_data.xlsx.parquet")
display(df_returns)

StatementMeta(, 1dc837c0-cb6e-40a0-b681-c50caec892b7, 22, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, e6be3055-89c2-439b-beea-a6c76cd62da2)

In [52]:
from pyspark.sql.functions import *

df_orders1 = (
    df_orders
    .withColumnRenamed("cust_id", "Customer_ID")
    .withColumnRenamed("Order_Amount$", "Order_Amount")

    .withColumn(
        "Order_ID",
        trim(col("Order_ID").cast("string"))
    )

    .withColumn(
        "Customer_ID",
        upper(trim(col("Customer_ID").cast("string")))
    )

    .withColumn(
        "Product_Name",
        initcap(trim(col("Product_Name")))
    )

    # --------------------------------------------------
# 5. Clean Quantity
# --------------------------------------------------
.withColumn(
    "Qty",
    when(
        lower(trim(col("Qty").cast("string"))).isin("one", "1"),
        lit(1)
    )
    .when(
        lower(trim(col("Qty").cast("string"))).isin("two", "2"),
        lit(2)
    )
    .when(
        lower(trim(col("Qty").cast("string"))).isin("three", "3"),
        lit(3)
    )
    .when(
        lower(trim(col("Qty").cast("string"))).isin("four", "4"),
        lit(4)
    )
    .when(
        lower(trim(col("Qty").cast("string"))).isin("five", "5"),
        lit(5)
    )
    .when(
        lower(trim(col("Qty").cast("string"))).isin("nan", "null", ""),
        lit(None)
    )
    .otherwise(
        col("Qty").cast("int")
    )
)

    # DATE CLEANING
    .withColumn(
        "Order_Date",
        coalesce(
            to_date(col("Order_Date"), "yyyy-MM-dd"),
            to_date(col("Order_Date"), "yyyy/MM/dd"),
            to_date(col("Order_Date"), "yyyy.MM.dd"),
            to_date(col("Order_Date"), "dd-MM-yyyy"),
            to_date(col("Order_Date"), "dd/MM/yyyy"),
            to_date(col("Order_Date"), "dd.MM.yyyy"),
            to_date(col("Order_Date"), "MM-dd-yyyy"),
            to_date(col("Order_Date"), "MM/dd/yyyy"),
            to_date(col("Order_Date"), "MM.dd.yyyy"),
             to_date(
            regexp_replace(
                trim(col("Order_Date").cast("string")),
                r"(\d+)(st|nd|rd|th)",r"$1"),"MMMM d yyyy"),
            to_date(col("Order_Date"), "yyyy/dd/MM")
        )
    )

    # --------------------------------------------------
    # 7. Clean Order Amount
    # --------------------------------------------------
    .withColumn(
        "Order_Amount",
        regexp_extract(
            col("Order_Amount").cast("string"),
            r"(\d+(?:\.\d+)?)",
            1
        ).cast("double")
    )

    .withColumn(
        "Delivery_Status",
        initcap(trim(col("Delivery_Status")))
    )

    .withColumn(
        "Payment_Mode",
        when(
            col("Payment_Mode").isNull() |
            (trim(col("Payment_Mode")) == ""),
            "Unknown"
        ).otherwise(
            initcap(trim(col("Payment_Mode")))
        )
    )

    .withColumn(
        "Ship_Address",
        trim(col("Ship_Address"))
    )

    .withColumn(
        "Email",
        lower(trim(col("Email")))
    )

    .withColumn(
        "Promo_Code",
        when(
            col("Promo_Code").isNull() |
            (trim(col("Promo_Code")) == ""),
            "NO_PROMO"
        ).otherwise(
            upper(trim(col("Promo_Code")))
        )
    )

    .withColumn(
        "Feedback_Score",
        col("Feedback_Score").cast("double")
    )

    .withColumn(
        "Order_Year",
        year(col("Order_Date"))
    )

    .withColumn(
        "Order_Month",
        month(col("Order_Date"))
    )
)

display(df_orders1)

StatementMeta(, 1dc837c0-cb6e-40a0-b681-c50caec892b7, 23, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 5162cfda-218b-4833-b8ce-9ee72700bc67)

In [44]:
df_orders1.write.mode("overwrite").format("delta").saveAsTable("silver_orders")

StatementMeta(, 1dc837c0-cb6e-40a0-b681-c50caec892b7, 24, Finished, Available, Finished, False)

In [30]:
from pyspark.sql.functions import *

df_inventory = (
    df_invent

    # 1. Standardize column names
    .withColumnRenamed("productName", "ProductName")
    .withColumnRenamed("cost_price", "CostPrice")
    .withColumnRenamed("last_stocked", "LastStocked")

    # 2. Clean Stock
    .withColumn(
        "Stock",
        when(
            trim(col("stock")).rlike("^[0-9]+$"),
            col("stock").cast("int")
        )
        .when(
            col("stock").isNull() | (trim(col("stock")) == ""),
            lit(None).cast("int")
        )
        .otherwise(
            when(lower(trim(col("stock"))).contains("twenty five"), lit(25))
            .when(lower(trim(col("stock"))).contains("twenty"), lit(20))
            .when(lower(trim(col("stock"))).contains("eighteen"), lit(18))
            .when(lower(trim(col("stock"))).contains("fifteen"), lit(15))
            .when(lower(trim(col("stock"))).contains("twelve"), lit(12))
            .otherwise(lit(None).cast("int"))
        )
    )

    # 3. Clean LastStocked - handle multiple formats
    .withColumn(
        "LastStocked",
        coalesce(
            to_date(trim(col("LastStocked").cast("string")), "yyyy-MM-dd"),
            to_date(trim(col("LastStocked").cast("string")), "yyyy/MM/dd"),
            to_date(trim(col("LastStocked").cast("string")), "yyyy.MM.dd"),
            to_date(trim(col("LastStocked").cast("string")), "dd-MM-yyyy"),
            to_date(trim(col("LastStocked").cast("string")), "dd/MM/yyyy"),
            to_date(trim(col("LastStocked").cast("string")), "dd.MM.yyyy"),
            to_date(trim(col("LastStocked").cast("string")), "MM/dd/yyyy")
        )
    )

    # 4. Clean CostPrice
    .withColumn(
        "CostPrice",
        regexp_extract(
            col("CostPrice").cast("string"),
            r"(\d+\.?\d*)",
            1
        ).cast("double")
    )

    # 5. Clean Warehouse
    .withColumn(
        "Warehouse",
        initcap(
            trim(
                regexp_replace(
                    col("warehouse"),
                    r"[^a-zA-Z0-9\s]",
                    " "
                )
            )
        )
    )

    # 6. Standardize Available
    .withColumn(
        "Available",
        when(
            lower(trim(col("available"))).isin(
                "yes", "y", "true"
            ),
            lit(True)
        )
        .when(
            lower(trim(col("available"))).isin(
                "no", "n", "false"
            ),
            lit(False)
        )
        .otherwise(lit(None).cast("boolean"))
    )

    # 7. Remove raw columns
    .drop(
        "stock",
        "warehouse",
        "available"
    )
)

display(df_inventory)

StatementMeta(, 1dc837c0-cb6e-40a0-b681-c50caec892b7, 25, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, ae90f13f-cd99-4e6b-bd3d-8a2f084062bd)

In [45]:
df_inventory.write.mode("overwrite").format("delta").saveAsTable("silver_inventory")


StatementMeta(, 1dc837c0-cb6e-40a0-b681-c50caec892b7, 26, Finished, Available, Finished, False)

In [32]:
from pyspark.sql.functions import *
from pyspark.sql.types import DoubleType

df_return_cleaned = (
    df_returns

    # --------------------------------------------------
    # 1. Rename Prop columns to actual column names
    # --------------------------------------------------
    .withColumnRenamed("Prop_0", "Return_ID")
    .withColumnRenamed("Prop_1", "Order_ID")
    .withColumnRenamed("Prop_2", "Customer_ID")
    .withColumnRenamed("Prop_3", "Product")
    .withColumnRenamed("Prop_4", "Return_Reason")
    .withColumnRenamed("Prop_5", "Return_Date")
    .withColumnRenamed("Prop_6", "Refund_Status")
    .withColumnRenamed("Prop_7", "Pickup_Address")
    .withColumnRenamed("Prop_8", "Return_Amount")

    # --------------------------------------------------
    # 2. Remove the header row if it was loaded as data
    # --------------------------------------------------
    .filter(col("Return_ID") != "Return_ID")

    # --------------------------------------------------
    # 3. Standardize column names
    # --------------------------------------------------
    .withColumnRenamed("Return_ID", "ReturnID")
    .withColumnRenamed("Order_ID", "OrderID")
    .withColumnRenamed("Customer_ID", "CustomerID")
    .withColumnRenamed("Return_Reason", "ReturnReason")
    .withColumnRenamed("Return_Date", "ReturnDate")
    .withColumnRenamed("Refund_Status", "RefundStatus")
    .withColumnRenamed("Pickup_Address", "PickupAddress")
    .withColumnRenamed("Return_Amount", "ReturnAmount")

    # --------------------------------------------------
    # 4. Clean Return Date
    # Multiple formats exist in your source
    # --------------------------------------------------
    .withColumn(
        "ReturnDate",
        coalesce(
            to_date(trim(col("ReturnDate")), "yyyy-MM-dd"),
            to_date(trim(col("ReturnDate")), "yyyy/MM/dd"),
            to_date(trim(col("ReturnDate")), "yyyy.MM.dd"),
            to_date(trim(col("ReturnDate")), "dd-MM-yyyy"),
            to_date(trim(col("ReturnDate")), "dd/MM/yyyy"),
            to_date(trim(col("ReturnDate")), "dd.MM.yyyy"),
            to_date(trim(col("ReturnDate")), "MM/dd/yyyy"),
            to_date(trim(col("ReturnDate")), "MMMM d yyyy")
        )
    )

    # --------------------------------------------------
    # 5. Clean Refund Status
    # --------------------------------------------------
    .withColumn(
        "RefundStatus",
        lower(
            regexp_replace(
                trim(col("RefundStatus")),
                r"[^a-zA-Z]",
                ""
            )
        )
    )

    # --------------------------------------------------
    # 6. Clean Return Amount
    # --------------------------------------------------
    .withColumn(
        "ReturnAmount",
        regexp_extract(
            col("ReturnAmount").cast("string"),
            r"(\d+\.?\d*)",
            1
        ).cast(DoubleType())
    )

    # --------------------------------------------------
    # 7. Clean Pickup Address
    # --------------------------------------------------
    .withColumn(
        "PickupAddress",
        initcap(
            trim(
                regexp_replace(
                    col("PickupAddress"),
                    r"[^a-zA-Z0-9\s]",
                    " "
                )
            )
        )
    )

    # --------------------------------------------------
    # 8. Clean Product
    # --------------------------------------------------
    .withColumn(
        "Product",
        initcap(
            trim(
                regexp_replace(
                    col("Product"),
                    r"[^a-zA-Z0-9\s]",
                    ""
                )
            )
        )
    )

    # --------------------------------------------------
    # 9. Clean Customer ID
    # --------------------------------------------------
    .withColumn(
        "CustomerID",
        upper(trim(col("CustomerID")))
    )

    # --------------------------------------------------
    # 10. Remove records without Return ID
    # --------------------------------------------------
    .filter(col("ReturnID").isNotNull())

    # --------------------------------------------------
    # 11. Remove duplicate Return IDs
    # --------------------------------------------------
    .dropDuplicates(["ReturnID"])
)

display(df_return_cleaned)

StatementMeta(, 1dc837c0-cb6e-40a0-b681-c50caec892b7, 27, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, deb4a309-c1e2-45f0-a600-267e3153859b)

In [46]:
df_return_cleaned.write.mode("overwrite").format("delta").saveAsTable("silver_returns")


StatementMeta(, 1dc837c0-cb6e-40a0-b681-c50caec892b7, 28, Finished, Available, Finished, False)

In [47]:
from pyspark.sql.functions import *

# Load Silver Tables
df_orders = spark.read.table("silver_orders")
df_returns = spark.read.table("silver_returns")
df_inventory = spark.read.table("silver_inventory")


StatementMeta(, 1dc837c0-cb6e-40a0-b681-c50caec892b7, 29, Finished, Available, Finished, False)

In [54]:
df_order_return = df_orders.join(
    df_returns,
    df_orders["Order_ID"] == df_returns["OrderID"],
    "left"
)
display(df_order_return)

StatementMeta(, 1dc837c0-cb6e-40a0-b681-c50caec892b7, 30, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, e130ce6d-4230-43c3-9bf4-720c71bb6b15)

In [57]:
df_enriched = df_order_return.join(
    df_inventory,
    df_order_return.Product_Name == df_inventory.ProductName,
    "left"
)
display(df_enriched)

StatementMeta(, 1dc837c0-cb6e-40a0-b681-c50caec892b7, 31, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, d513ad4b-0807-47ca-bfbf-7613d54f24bd)

In [ ]:
from pyspark.sql import functions as F

df_kpi = (
    df_enriched
    .groupBy("Product_Name")
    .agg(

        # Total unique orders
        F.countDistinct("Order_ID").alias("Total_Orders"),

        # Unique customers
        F.countDistinct("Customer_ID").alias("Unique_Customers"),

        # Total returns
        F.countDistinct("ReturnID").alias("Total_Returns"),

        # Return rate
        F.round(
            F.countDistinct("ReturnID") /
            F.countDistinct("Order_ID") * 100,
            2
        ).alias("Return_Rate_Percent"),

        # Revenue
        F.round(
            F.sum("Order_Amount"),
            2
        ).alias("Total_Revenue"),

        # Average order value
        F.round(
            F.avg("Order_Amount"),
            2
        ).alias("Avg_Order_Value"),

        # Average cost
        F.round(
            F.avg("CostPrice"),
            2
        ).alias("Avg_Cost"),

        # Total return amount
        F.round(
            F.sum("ReturnAmount"),
            2
        ).alias("Total_Return_Amount")
    )
)

display(df_kpi)

StatementMeta(, 1dc837c0-cb6e-40a0-b681-c50caec892b7, 36, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 19900687-23bb-498f-82a9-16fe9f9714b8)

In [ ]:
df_kpi.write.mode("overwrite").format("delta").saveAsTable("gold_product_kpis")


StatementMeta(, 1dc837c0-cb6e-40a0-b681-c50caec892b7, 39, Finished, Available, Finished, False)